In [1]:
import json
import ctypes
import dask.dataframe as dd
import pandas as pd
from dask.distributed import Client


def trim_memory() -> int:
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)


def extract_helpful(df: dd.DataFrame, idx: int) -> dd.Series:
    def extract(x):
        split = str(x).strip("[]").split(", ")
        return split[idx] if len(split) > idx else "0"

    return df["helpful"].map_partitions(
        lambda part: part.apply(extract), meta=("helpful", "object")
    )


def extract_year(df: dd.DataFrame) -> dd.Series:
    return df["reviewTime"].map_partitions(
        lambda part: part.apply(
            lambda x: x.split(", ")[-1] if pd.notna(x) and ", " in x else None
        ),
        meta=("reviewing_since", "object"),
    )


def PA0(path_to_user_reviews_csv):
    client = Client()
    client.run(trim_memory)
    client = client.restart()

    user_reviews_dd = dd.read_csv(path_to_user_reviews_csv)

    user_reviews_dd["helpful_votes"] = extract_helpful(user_reviews_dd, 0).astype(int)
    user_reviews_dd["total_votes"] = extract_helpful(user_reviews_dd, 1).astype(int)
    user_reviews_dd["reviewing_since"] = extract_year(user_reviews_dd).astype(int)

    grouped = user_reviews_dd.groupby("reviewerID").agg(
        {
            "reviewerID": "count",
            "overall": "mean",
            "reviewing_since": "min",
            "helpful_votes": "sum",
            "total_votes": "sum",
        }
    )

    grouped = grouped.rename(
        columns={
            "reviewerID": "number_products_rated",
            "overall": "avg_ratings",
        }
    ).reset_index()

    grouped = grouped.persist()
    submit = grouped.describe().compute().round(2)
    
    with open("results_PA0.json", "w") as outfile:
        json.dump(json.loads(submit.to_json()), outfile)

In [2]:
PA0('user_reviews.csv')

/home/ubuntu/dask_env/lib/python3.10/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 46703 instead
  warnings.warn(
2025-04-16 02:30:08,366 - distributed.nanny - WARNING - Restarting worker
2025-04-16 02:30:08,371 - distributed.nanny - WARNING - Restarting worker
2025-04-16 02:30:08,390 - distributed.nanny - WARNING - Restarting worker
2025-04-16 02:30:08,403 - distributed.nanny - WARNING - Restarting worker
2025-04-16 02:30:14,813 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 3be27387bcf0126ed51f14c38a726426 initialized by task ('shuffle-transfer-3be27387bcf0126ed51f14c38a726426', 105) executed on worker tcp://127.0.0.1:34403
2025-04-16 02:43:09,060 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 3be27387bcf0126ed51f14c38a726426 deactivated due to stimulus 'task-finished-1744771388.9993858'


In [52]:
!diff results_PA0.json expected_results_PA0.json

1c1
< {"number_products_rated": {"count": 453449.0, "mean": 1.09, "std": 0.73, "min": 1.0, "25%": 1.0, "50%": 1.0, "75%": 1.0, "max": 268.0}, "avg_ratings": {"count": 453449.0, "mean": 4.16, "std": 1.26, "min": 1.0, "25%": 4.0, "50%": 5.0, "75%": 5.0, "max": 5.0}, "reviewing_since": {"count": 453449.0, "mean": 2011.67, "std": 2.99, "min": 1997.0, "25%": 2011.0, "50%": 2013.0, "75%": 2013.0, "max": 2014.0}, "helpful_votes": {"count": 453449.0, "mean": 2.23, "std": 17.85, "min": 0.0, "25%": 0.0, "50%": 0.0, "75%": 1.0, "max": 8125.0}, "total_votes": {"count": 453449.0, "mean": 3.01, "std": 19.77, "min": 0.0, "25%": 0.0, "50%": 0.0, "75%": 2.0, "max": 8248.0}}
\ No newline at end of file
---
> {"number_products_rated": {"count": 16121597.0, "mean": 3.08, "std": 13.18, "min": 1.0, "25%": 1.0, "50%": 1.0, "75%": 3.0, "max": 26615.0}, "avg_ratings": {"count": 16121597.0, "mean": 4.09, "std": 1.21, "min": 1.0, "25%": 3.67, "50%": 4.6, "75%": 5.0, "max": 5.0}, "reviewing_since": {"count": 1612

In [35]:
import pandas as pd
pd.read_json('results_PA0.json')

,number_products_rated,avg_ratings,reviewing_since,helpful_votes,total_votes
count,16121597.00,16121597.00,16121597.00,16121597.00,16121597.00
mean,3.08,4.09,2011.05,6.37,8.58
std,13.18,1.21,3.39,87.56,102.08
min,1.00,1.00,1996.00,0.00,0.00
25%,1.00,3.67,2010.00,0.00,0.00
50%,1.00,4.60,2012.00,1.00,1.00
75%,3.00,5.00,2013.00,3.00,5.00
max,26615.00,5.00,2014.00,124429.00,149617.00
